Setup, Loading, & EDA

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import optuna
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
np.random.seed(42)


TARGET = 'health_condition'
ID = 'id'
CLASS_ORDER = ['fit', 'at-risk', 'unhealthy']

NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality','physical_activity_level', 'smoking_alcohol', 'gender']

DATA_DIR = Path("data") # local path

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"


train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

#print train and test shape
print("train: ", train.shape)
print("test: ", test.shape)

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True))



Preprocessing Pipeline

In [ ]:
# Prepare features
X = train.drop(columns=[TARGET, ID])
y = train[TARGET]
X_test = test.drop(columns=[ID])

# Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Convert string types to category so LightGBM handles them natively
for col in CAT_COLS:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

Hyperparameter Optimization (Optuna)

In [ ]:
# Stratified K folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def train_cv(params):
    cv_scores = []

    for train_idx, valid_idx in cv.split(X, y_encoded):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y_encoded[train_idx]
        y_valid = y_encoded[valid_idx]

        model = LGBMClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(100, verbose=False)
            ]
        )

        preds = model.predict(X_valid)

        score = balanced_accuracy_score(
            y_valid,
            preds
        )

        cv_scores.append(score)

    return np.mean(cv_scores)

def objective(trial):
    params = {
        "objective": "multiclass",
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            200
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            5,
            100
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10,
            log=True
        ),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }

    return train_cv(params)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print(study.best_value)
print(study.best_params)

Cross-Validation Loop with Best Hyperparameters

In [ ]:
# get best params
best_params = study.best_params
best_params.update({
    "objective": "multiclass",
    "n_estimators": 1000,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
})


n_classes = len(le.classes_)

oof_probs = np.zeros((len(X), n_classes))
test_probs = np.zeros((len(X_test), n_classes))

cv_scores = []
models = []

# CV loop
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y_encoded), start=1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y_encoded[train_idx]
    y_valid = y_encoded[valid_idx]


    model = LGBMClassifier(**best_params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(100, verbose=False)
        ]
    )

    val_probs = model.predict_proba(X_valid)
    oof_probs[valid_idx] = val_probs

    test_probs += model.predict_proba(X_test) / cv.n_splits

    preds = val_probs.argmax(axis=1)

    score = balanced_accuracy_score(y_valid, preds)

    cv_scores.append(score)

    print(f"Fold {fold}: {score:.5f}")

    models.append(model)


overall_score = balanced_accuracy_score(
    y_encoded,
    oof_probs.argmax(axis=1)
)

print("=" * 50)
print(f"CV Scores: {cv_scores}")
print(f"Mean CV: {np.mean(cv_scores):.5f}")
print(f"Std CV : {np.std(cv_scores):.5f}")
print(f"OOF BA : {overall_score:.5f}")


Final Evaluation & Submission

In [ ]:
# Plot Confusion Matrix
oof_preds = oof_probs.argmax(axis=1)
cm = confusion_matrix(y_encoded, oof_preds, normalize="true")

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix (OOF)")
plt.tight_layout()
plt.show()

# Plot Top Feature Importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": np.mean([m.feature_importances_ for m in models], axis=0) # Average across folds
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance.head(20)["feature"], importance.head(20)["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Average Feature Importance")
plt.title("LightGBM Feature Importance (Mean across folds)")
plt.tight_layout()
plt.show()

# Create Submission File
test_pred = test_probs.argmax(axis=1)
submission = pd.DataFrame({
    "id": test[ID],
    "target": le.inverse_transform(test_pred)
})
submission.to_csv("submission.csv", index=False)